## Hypotesis testing

- **Hypothesis testing:**
  - Decide between two opposite statements about a population. These statements are called:
    - Null hypothesis: H0
    - Alternative hypothesis: H1/a
- **Steps to do a statistical test:**
  - Decide which is the right statistical test (what do you want to evaluate)
  - Set your significance level (alpha = 0.05, 0.01 for medical purposes)
  - Collect a sample (your dataset)
  - Compute the test statistic (different for each test)
  - Resolve the test:
    - Compare the value of your test_statistic vs critical value/s (the distribution is different for each test)
    - Computing the p-value, and comparing it with the significance level: alpha
      - If p-value = P( |s| >= statistic | H0 == True ) < alpha → Reject H0

### Set the hypothesis:  #1
Hypothesis testing: The UX change cause more clients to complete the process as the % of Test group finishers is 5.6% more than Control group finishers. 

H0: The completion rate of Test ≤ completion rate of Control (the UX change had no positive effect)


H1: The completion rate of Test > completion rate of Control

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
%matplotlib inline

In [2]:
clean_df=pd.read_csv("../data/clean/error_compRate_clean.csv", sep=None, engine="python")
df=clean_df.copy()
display(df.head(2))

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,visitor_id,visit_id,process_step,date_time,step_num,step_diff,is_error,attempt_No,finish
0,169,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,no_data,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,0,NaN,False,1,False
1,169,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,no_data,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,1,1.0,False,1,False


In [3]:
#Are the groups independent?:
control_ids = set(df[df.Variation=='Control']['client_id'].unique())
test_ids = set(df[df.Variation=='Test']['client_id'].unique())

overlap = control_ids & test_ids
print(f'Overlapping clients: {len(overlap)}')

Overlapping clients: 0


In [4]:
#Setting significance level
alpha = 0.05
#Collect a samples
control_finishers =df[(df.Variation=='Control') & (df.finish==True)].client_id.nunique()
test_finishers =df[(df.Variation=='Test') & (df.finish==True)].client_id.nunique()

# total clients per group
control_total = df[df.Variation=='Control']['client_id'].nunique()
test_total = df[df.Variation=='Test']['client_id'].nunique()


In [5]:
from scipy.stats import norm

# compute proportions
p_test = test_finishers / test_total
p_control = control_finishers / control_total
p_combined = (test_finishers + control_finishers) / (test_total + control_total)

# z-stat
se = np.sqrt(p_combined * (1 - p_combined) * (1/test_total + 1/control_total))
z_stat = (p_test - p_control) / se

# one-sided p-value
p_value = 1 - norm.cdf(z_stat)

print(f'z-stat: {z_stat:.4f}')
print(f'p-value: {p_value:.4f}')

if p_value< alpha:
    print(f'As p_value is < 0.05, the H0 is rejected, meaning that the completion rate of Test group is greater as the\n completion rate of Control ')
else:
    print('As p_value is p_value > 0.05, the H0 is accepted, meaning that the completion rate of Test group is less or equal as the\n completion rate of Control')

z-stat: 13.1564
p-value: 0.0000
As p_value is < 0.05, the H0 is rejected, meaning that the completion rate of Test group is greater as the
 completion rate of Control 


### Set the hypothesis:  #2

Hypothesis testing: The error ratio is greater in test group than to control group 

H0: The error ratio in test group is <= than control group 

H1: The error ratio in test group is greater than control group (sample_control_error_ratio >= sample_test_error_ratio)

In [6]:

# errors per group (steps flagged as backward-nav)
control_errors = df[df.Variation=='Control']['is_error'].sum()
test_errors = df[df.Variation=='Test']['is_error'].sum()

# all steps where going back was possible, for both groups only 1,2,3,4
control_steps = df[(df.Variation=='Control') & (df.step_num.isin([1,2,3,4]))].shape[0]
test_steps = df[(df.Variation=='Test') & (df.step_num.isin([1,2,3,4]))].shape[0]


In [7]:
# compute proportions
p_test = test_errors / test_steps
p_control = control_errors / control_steps
p_combined = (test_errors + control_errors) / (test_steps + control_steps)

# z-stat
se = np.sqrt(p_combined * (1 - p_combined) * (1/test_steps + 1/control_steps))
z_stat = (p_test - p_control) / se

# one-sided p-value
p_value = 1 - norm.cdf(z_stat)

print(f'p_test (error rate):    {p_test:.4f}')
print(f'p_control (error rate): {p_control:.4f}')
print(f'z-stat:  {z_stat:.4f}')
print(f'p-value: {p_value:.4f}')

if p_value< alpha:
    print('As p_value is < 0.05, the H0 is rejected, meaning that the error ratio in test group is > than control group')
else:
    print('As p_value is p_value > 0.05, the H0 is accepted, meaning that the error ratio in test group is <= than control group')

p_test (error rate):    0.1357
p_control (error rate): 0.1021
z-stat:  23.7421
p-value: 0.0000
As p_value is < 0.05, the H0 is rejected, meaning that the error ratio in test group is > than control group
